In [1]:
import pandas as pd
import json
import re
import networkx as nx
from pyvis.network import Network

# Функції для перерахунку та очищення
def calculate_criticality_score(event_type: str, is_confirmed: bool, impact_text: str) -> int:
    impact_lower = str(impact_text).lower()
    event_lower = str(event_type).lower()
    score = 1

    if event_lower == "physical_impact": score = 10 if is_confirmed else 7
    elif event_lower == "data_destruction": score = 9 if is_confirmed else 5
    elif event_lower == "ransomware": score = 7 if is_confirmed else 4
    elif event_lower == "data_exfiltration": score = 6 if is_confirmed else 3
    elif event_lower == "unauthorized_access": score = 5 if is_confirmed else 2
    elif event_lower == "malware": score = 4 if is_confirmed else 1
    elif event_lower == "ddos": score = 3 if is_confirmed else 1
    elif event_lower == "defacement": score = 2 if is_confirmed else 1
    elif event_lower in ["phishing", "malware_distribution", "information_operation"]: score = 1

    if is_confirmed:
        physical_keywords = ["scada", "асутп", "плк", "plc", "rtu", "обладнан", "зупинк", "блекаут", "знеструмлен", "технологічн", "виробництв"]
        if any(word in impact_lower for word in physical_keywords):
            score = max(score, 8)
    return max(0, min(10, score))

def clean_threat_actor(actor):
    if not actor or str(actor).lower() in ['nevdo_mo', 'nevіdомо', 'зловмисник', 'зловмисники']:
        return 'невідомо'
    match = re.search(r'(UAC-\d{4})', str(actor), re.IGNORECASE)
    if match: return match.group(1).upper()
    return str(actor)

# Словники 
entity_mapping = {
    'ua-gov[.]cloud': 'невідомо', 'ukraer': 'невідомо', 'UAC-0180': 'невідомо',
    'UAC-0173': 'невідомо', 'RemcosRAT servers': 'невідомо', 'Тківський обслуговування': 'невідомо'
}

type_mapping = {
    'військове формування': 'сили безпеки та оборони',
    'EOM of defense industry employees and Ukrainian Armed Forces personnel': 'сили безпеки та оборони',
    'government': 'державна установа', 'фінанси': 'фінансова установа',
    'телефонізаційний оператор': 'телекомунікаційний оператор', 'Incident Report': 'невідомо',
    'Network': 'невідомо', 'domain': 'невідомо', 'Domain Name': 'невідомо', 'servers': 'невідомо',
    'cyber threat actor': 'невідомо', "комп'ютерний системи": 'невідомо',
    'постачальник рішень АСУТП': 'промислове підприємство', 'оборонно-промисловий комплекс': 'промислове підприємство',
    'индустріальна установа': 'промислове підприємство', 'школа/університет': 'навчальний заклад',
    'Individuals': 'громадяни', 'diplomatic institution': 'дипломатична установа',
    'комунікальне підприємство': 'комунальне підприємство', 'Corporate network': 'комерційна організація',
    'Individual/Company': 'комерційна організація', 'компанія': 'комерційна організація', 'ІТ компанія': 'комерційна організація'
}


# Підписи секторів 
SECTOR_LABELS = {
    'government': 'державний сектор', 'telecom': 'телеком', 'defense': 'оборона та військові',
    'finance': 'фінанси', 'industrial': 'промисловість', 'transport': 'транспорт',
    'media': 'медіа', 'energy': 'енергетика', 'justice': 'юстиція та суди',
    'healthcare': "охорона здоров'я", 'water_supply': 'водопостачання',
    'unknown': 'сектор невідомо'
}

def get_sector_label(sector):
    key = str(sector).strip().lower() if sector is not None else 'unknown'
    return SECTOR_LABELS.get(key, 'сектор невідомо')


events = []
original_data = []

with open('enriched_events.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        ev = json.loads(line)
        events.append(ev)
        if 'llm_extraction' in ev:
            original_data.append(dict(ev['llm_extraction']))

df_before = pd.DataFrame(original_data)


cleaned_data = []

for event in events:
    if 'llm_extraction' in event:
        ext = event['llm_extraction']
        
        if ext.get('target_entity') == 'Banking Institution':
            ext['target_type'] = 'фінансова установа'
            ext['target_entity'] = 'Банківська установа'  
        elif ext.get('target_entity') == 'Нотаріат України':
            ext['target_type'] = 'державна установа'
            ext['target_entity'] = 'Нотаріат України'
        elif ext.get('target_entity') == 'Укрзалізниця':
            ext['target_type'] = 'державна установа'
            ext['target_entity'] = 'Укрзалізниця'
        elif ext.get('target_entity') in ['Military personnel', 'ЕОМ представника Сил оборони України', 'Сил оборони України']:
            ext['target_type'] = 'сили безпеки та оборони'
            ext['target_entity'] = 'Сили оборони України'
        elif ext.get('target_entity') == 'UAC (Український світовий конґрес)':
            ext['target_entity'] = 'Світовий конґрес українців'
        elif str(ext.get('target_entity')).lower() in ['державні органи україни', 'державна установа']:
            ext['target_type'] = 'державна установа'
            ext['target_entity'] = 'невідомо'  
        else:
            if ext.get('target_entity') in entity_mapping:
                ext['target_entity'] = entity_mapping[ext['target_entity']]
                
        if ext.get('target_type') in type_mapping:
            ext['target_type'] = type_mapping[ext['target_type']]
            
        ext['threat_actor'] = clean_threat_actor(ext.get('threat_actor'))

        # Розрахунок КРИТИЧНОСТІ замість ризику
        e_type = ext.get('event_type', 'other')
        is_conf = ext.get('is_impact_confirmed', False)
        imp_text = ext.get('confirmed_impact', '')
        
        new_score = calculate_criticality_score(e_type, is_conf, imp_text)
        
        
        ext['criticality_score'] = new_score
        if 'preliminary_impact_score' in ext:
            del ext['preliminary_impact_score']
        
        cleaned_data.append(ext)

df_after = pd.DataFrame(cleaned_data)

columns_to_show = ['target_entity', 'target_type']

for col in columns_to_show:
    print(f"\n{'='*60}")
    print(f"ПОРІВНЯННЯ ДЛЯ ПОЛЯ: {col.upper()}")
    print(f"{'='*60}")
    
    print("\nДО ОЧИЩЕННЯ (Топ 10):")
    print("-" * 30)
    print(df_before[col].value_counts().head(10))
    
    print("\nПІСЛЯ ОЧИЩЕННЯ (Топ 10):")
    print("-" * 30)
    print(df_after[col].value_counts().head(10))

output_filename = 'cleaned_events.jsonl'
with open(output_filename, 'w', encoding='utf-8') as f:
    for event in events:
        f.write(json.dumps(event, ensure_ascii=False) + '\n')

print(f"\nДані очищено та збережено у '{output_filename}'!")

print("Завантаження даних та побудова графа...")

graph_events = []
with open('cleaned_events.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        graph_events.append(json.loads(line)['llm_extraction'])

df_graph = pd.DataFrame(graph_events)
df_graph = df_graph[df_graph['threat_actor'] != 'невідомо']

G = nx.Graph()
edge_criticality_map = {}  
edge_counter = 0

for _, row in df_graph.iterrows():
    actor = row['threat_actor']
    target_type = row['target_type']
    sector_label = get_sector_label(row.get('sector'))
    sector_node = f"SECTOR::{sector_label}"
    target_entity = row['target_entity']
    
    vector = row.get('attack_vector', 'unknown')
    e_type = row.get('event_type', 'unknown')
    criticality = row.get('criticality_score', 1) 
    
    G.add_node(actor, node_type='actor')
    G.add_node(sector_node, node_type='sector', label=sector_label)
    
    incident_detail = f"• [{e_type}] Вектор: {vector} | Тип цілі: {target_type} | Бал критичності: {criticality}/10"
    
    if target_entity == 'невідомо':
        if G.has_edge(actor, sector_node):
            G[actor][sector_node]['weight'] += 1
            G[actor][sector_node]['criticalities'].append(criticality) # Оновлено
            G[actor][sector_node]['details'].append(incident_detail)
        else:
            G.add_edge(actor, sector_node, weight=1, 
                       criticalities=[criticality], details=[incident_detail])
    else:
        G.add_node(target_entity, node_type='entity')
        
        if G.has_edge(actor, target_entity):
            G[actor][target_entity]['weight'] += 1
            G[actor][target_entity]['criticalities'].append(criticality)
            G[actor][target_entity]['details'].append(incident_detail)
        else:
            G.add_edge(actor, target_entity, weight=1, 
                       criticalities=[criticality], details=[incident_detail])
            
        if not G.has_edge(target_entity, sector_node):
            G.add_edge(target_entity, sector_node, weight=1, is_structural=True)

net = Network(height='100vh', width='100%', bgcolor='#1a1a1a', font_color='white', notebook=True, cdn_resources='remote')
net.force_atlas_2based(spring_length=150, overlap=0)

for node, node_attrs in G.nodes(data=True):
    degree = G.degree(node)
    n_type = node_attrs.get('node_type')
    display_label = node_attrs.get('label', node)
    
    if n_type == 'actor':
        color, size, title = '#ff4b4b', 15 + (degree * 5), "Хакерська група (APT)"
    elif n_type == 'sector':
        color, size, title = '#4b8bff', 20 + (degree * 4), "Сектор критичної інфраструктури"
    elif n_type == 'entity':
        color, size, title = '#10b981', 15 + (degree * 3), "Конкретна організація"
        
    net.add_node(node, label=display_label, color=color, size=size, title=title)

for u, v, edge_attrs in G.edges(data=True):
    edge_counter += 1
    edge_id = f"edge_{edge_counter}"
    
    if edge_attrs.get('is_structural'):
        net.add_edge(u, v, id=edge_id, value=1, title="Належність до сектора", color='#444444', dashes=True)
    else:
        w = edge_attrs.get('weight', 1)
        criticalities = edge_attrs.get('criticalities', [1])
        
        max_criticality = max(criticalities)
        details_list = edge_attrs.get('details', [])
        
        edge_criticality_map[edge_id] = max_criticality
        
        formatted_details = "\n".join(details_list)
        
        tooltip = (f"Кількість інцидентів: {w}\n"
                   f"Макс. бал критичності: {max_criticality} / 10\n"
                   f"-----------------------------------\n"
                   f"Деталі інцидентів:\n{formatted_details}")
                   
        if max_criticality >= 8:
            edge_color = '#ff0000'
        elif max_criticality >= 5:
            edge_color = '#ffa500'
        else:
            edge_color = '#666666'
            
        net.add_edge(u, v, id=edge_id, value=w, title=tooltip, color=edge_color)

graph_filename = "final_diploma_graph.html"
net.show(graph_filename)

with open(graph_filename, 'r', encoding='utf-8') as f:
    html_content = f.read()


edge_criticality_json = json.dumps(edge_criticality_map)


control_panel_code = """
<div id="legend-panel" style="position: fixed; top: 20px; left: 20px; background: rgba(30,30,30,0.92); padding: 15px 20px; border-radius: 10px; color: #fff; z-index: 9999; font-family: Arial, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.5); border: 1px solid #444; width: 280px; backdrop-filter: blur(5px);">
    <h3 style="margin-top: 0; font-size: 15px; border-bottom: 1px solid #555; padding-bottom: 8px; color: #4b8bff;">Легенда та Фільтри</h3>
    
    <div style="margin-bottom: 12px; font-size: 12px; line-height: 1.5;">
        <b>Вузли:</b><br>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 10px; height: 10px; background: #ff4b4b; border-radius: 50%; display: inline-block; margin-right: 8px;"></span> Хакерська група (APT)</div>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 10px; height: 10px; background: #4b8bff; border-radius: 50%; display: inline-block; margin-right: 8px;"></span> Сектор критичної інфраструктури</div>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 10px; height: 10px; background: #10b981; border-radius: 50%; display: inline-block; margin-right: 8px;"></span> Конкретна організація</div>
    </div>

    <div style="margin-bottom: 12px; font-size: 12px; line-height: 1.5;">
        <b>Рівень критичності зв'язків (за макс. балом):</b><br>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 18px; height: 3px; background: #ff0000; display: inline-block; margin-right: 8px;"></span> Критичний (>= 8)</div>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 18px; height: 3px; background: #ffa500; display: inline-block; margin-right: 8px;"></span> Середній (5 - 7.9)</div>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 18px; height: 3px; background: #666666; display: inline-block; margin-right: 8px;"></span> Низький (< 5)</div>
        <div style="display: flex; align-items: center; margin-top: 3px;"><span style="width: 18px; height: 2px; border-top: 2px dashed #444; display: inline-block; margin-right: 8px;"></span> Структурний зв'язок</div>
    </div>

    <div style="font-size: 12px;">
        <label for="criticalityFilter" style="display: block; margin-bottom: 5px;"><b>Фільтр мін. балу критичності:</b></label>
        <select id="criticalityFilter" onchange="applyCriticalityFilter()" style="width: 100%; background: #222; color: #fff; border: 1px solid #555; padding: 6px; border-radius: 4px; cursor: pointer; outline: none;">
            <option value="0">Усі зв'язки (0+)</option>
            <option value="1">Від 1 і вище</option>
            <option value="2">Від 2 і вище</option>
            <option value="3">Від 3 і вище</option>
            <option value="4">Від 4 і вище</option>
            <option value="5">Від 5 і вище (Середній)</option>
            <option value="6">Від 6 і вище</option>
            <option value="7">Від 7 і вище</option>
            <option value="8">Від 8 і вище (Критичний)</option>
            <option value="9">Від 9 і вище</option>
            <option value="10">Лише рівень 10</option>
        </select>
    </div>
</div>

<script type="text/javascript">
    var edgeCriticalities = __EDGE_CRITICALITY_JSON__;

    function applyCriticalityFilter() {
        var minCrit = parseFloat(document.getElementById('criticalityFilter').value);
        var allEdges = edges.get();
        
        var updates = [];
        allEdges.forEach(function(edge) {
            if (edge.id in edgeCriticalities) {
                var crit = edgeCriticalities[edge.id];
                if (crit < minCrit) {
                    updates.push({id: edge.id, hidden: true});
                } else {
                    updates.push({id: edge.id, hidden: false});
                }
            }
        });
        edges.update(updates);
    }
</script>
</body>
"""


control_panel_code = control_panel_code.replace('__EDGE_CRITICALITY_JSON__', edge_criticality_json)
html_content = html_content.replace("</body>", control_panel_code)

with open(graph_filename, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\nГраф збережено у файл '{graph_filename}'")


ПОРІВНЯННЯ ДЛЯ ПОЛЯ: TARGET_ENTITY

ДО ОЧИЩЕННЯ (Топ 10):
------------------------------
target_entity
невідомо                                80
UKR.NET                                  2
Military personnel                       1
Banking Institution                      1
ua-gov[.]cloud                           1
ukraer                                   1
UAC-0180                                 1
Сили оборони України                     1
ЕОМ представника Сил оборони України     1
Сил оборони України                      1
Name: count, dtype: int64

ПІСЛЯ ОЧИЩЕННЯ (Топ 10):
------------------------------
target_entity
невідомо                      87
Сили оборони України           4
UKR.NET                        2
Укрзалізниця                   1
Укрінформ                      1
Світовий конґрес українців     1
Банківська установа            1
Нотаріат України               1
Name: count, dtype: int64

ПОРІВНЯННЯ ДЛЯ ПОЛЯ: TARGET_TYPE

ДО ОЧИЩЕННЯ (Топ 10):
----------------------